# 01 — Why Naive RAG Fails: The No-Retrieval Baseline

## Why this notebook exists

You have an LLM and a pile of documents the model was never trained on — a company wiki, a product catalog, a stack of PDFs. You want the model to answer questions about them. The simplest possible approach is: **paste all the documents into the prompt and ask.** No vector database, no embeddings, nothing fancy.

This notebook builds exactly that, and shows where it breaks. It works beautifully when you already know which single document holds the answer. The moment you *don't* — which is the real situation — you're forced to send everything, every time. We'll measure what that costs: multiples more tokens, money, and latency for the identical answer, a hard ceiling once the corpus outgrows the context window, and the quality risk of burying the relevant fact among hundreds of irrelevant ones.

The fix is **retrieval**: find the few relevant pieces and send only those. By the end of this notebook you'll feel exactly why that matters — which is the foundation the rest of the series builds on.

Requires an `OPENAI_API_KEY`. Our corpus describes a *fictional* company, so the model can't cheat from memory — every correct answer has to come from the context we provide.

## What you'll learn

- How "naive RAG" — putting documents directly in the prompt — works, and the one situation where it's perfectly fine.
- Why **not knowing which document is relevant** forces you to send the whole corpus on every query.
- How to **measure the cost of stuffing**: token counts (with `tiktoken`), dollar cost, and latency — and see the same answer cost multiples more.
- The **hard ceiling**: how to compute when a growing corpus simply won't fit in the model's context window, and what stuffing would cost per query at scale.
- The **"lost in the middle"** quality risk of long, mostly-irrelevant contexts.
- Why the answer is **retrieval** — send only the relevant pieces — which motivates embeddings (notebook 02).

## 1. Setup

This notebook calls the OpenAI chat API and counts tokens with `tiktoken`. We load `OPENAI_API_KEY` from the environment (and from a local `.env` file if `python-dotenv` is installed — real environment variables always win).

The guard cell below stops with a clear message if the key is missing. After it, we set up the OpenAI client, a `count_tokens` helper, illustrative pricing constants, and an `ask(question, context)` helper that answers a question using only the supplied context and reports how many tokens, dollars, and seconds it took.

In [ ]:
import os
import time

# Optional: load a local .env if python-dotenv is installed. Real env vars win.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── Key guard ──────────────────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    print("=" * 60)
    print("OPENAI_API_KEY is not set.")
    print("=" * 60)
    print()
    print("Every notebook in the RAG series calls the OpenAI API")
    print("(this one for the chat model; later ones for embeddings too).")
    print()
    print("Set it and restart the kernel:")
    print("  export OPENAI_API_KEY=sk-...")
    raise SystemExit("Set OPENAI_API_KEY and restart the kernel to continue.")

print("OPENAI_API_KEY set ✓")

In [ ]:
import tiktoken
from openai import OpenAI

DEFAULT_MODEL = "gpt-4o-mini"

# gpt-4o-mini pricing (illustrative — verify current rates at openai.com/pricing).
PRICE_INPUT_PER_1M = 0.15    # USD per 1M input (prompt) tokens
PRICE_OUTPUT_PER_1M = 0.60   # USD per 1M output (completion) tokens

openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment

_enc = tiktoken.encoding_for_model(DEFAULT_MODEL)


def count_tokens(text: str) -> int:
    """Count tokens the way the model will, using tiktoken."""
    return len(_enc.encode(text))


def ask(question: str, context: str, model: str = DEFAULT_MODEL) -> dict:
    """Answer `question` using ONLY `context`.

    Returns a dict with the answer plus token/latency/cost telemetry so we can
    measure what each call costs.
    """
    system = (
        "You are a helpful assistant. Answer the question using ONLY the provided "
        "context. If the answer is not in the context, say you don't know. Be concise."
    )
    user = f"Context:\n{context}\n\nQuestion: {question}"
    start = time.time()
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0,
    )
    latency = time.time() - start
    usage = resp.usage
    cost = (
        usage.prompt_tokens * PRICE_INPUT_PER_1M
        + usage.completion_tokens * PRICE_OUTPUT_PER_1M
    ) / 1_000_000
    return {
        "answer": resp.choices[0].message.content.strip(),
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "latency_s": latency,
        "cost_usd": cost,
    }


print("Setup OK")
print(f"Model: {DEFAULT_MODEL}")
print(f"count_tokens('hello world') = {count_tokens('hello world')}")

## 2. The Corpus

Our corpus is a handful of short markdown documents about **Halcyon Robotics**, a fictional warehouse-automation company. Using a made-up company is deliberate: the model has never seen it, so it genuinely cannot answer without the documents — which is the whole point of retrieval-augmented generation.

The cell below loads every `.md` file in `rag/data/` and prints its token count. These same documents are reused throughout the series; notebook 03 adds a PDF and teaches loading and chunking properly.

In [ ]:
from pathlib import Path

# Resolve the data folder whether the kernel runs from rag/ or from the repo root.
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("rag/data")

corpus: dict[str, str] = {}
for path in sorted(DATA_DIR.glob("*.md")):
    corpus[path.name] = path.read_text(encoding="utf-8")

print(f"Loaded {len(corpus)} documents from {DATA_DIR}/:\n")
total = 0
for name, text in corpus.items():
    n = count_tokens(text)
    total += n
    print(f"  {name:<28} {n:>5} tokens")
print(f"  {'-' * 28} {'-' * 11}")
print(f"  {'TOTAL':<28} {total:>5} tokens")

## 3. Naive RAG That Works — When You Already Know the Relevant Doc

Here is the simplest "RAG" there is: take the one document that holds the answer, put it in the prompt, and ask. We ask about Halcyon's support hours — a fact that lives only in `support.md` and nowhere in the model's training data.

This works perfectly. The answer is correct, grounded, and cheap, because we sent exactly the right context and nothing else.

> **Gotcha:** This only works because *we* knew `support.md` was the relevant document. We did the retrieval by hand. The entire rest of this series is about doing that step automatically — because in the real world you have thousands of documents and no idea up front which one holds the answer.

In [ ]:
# A question answerable only from Halcyon's docs (the model can't know it).
question = "What are Halcyon Robotics' technical support hours?"

# Hand-pick the ONE relevant document.
relevant_doc = corpus["support.md"]
result = ask(question, context=relevant_doc)

print(f"Q: {question}\n")
print(f"A: {result['answer']}\n")
print(f"context tokens : {result['prompt_tokens']}")
print(f"latency        : {result['latency_s']:.2f}s")
print(f"cost           : ${result['cost_usd']:.6f}")

## 4. But You Don't Know Which Doc — So You Stuff Everything

In reality you can't hand-pick the relevant document, because you don't know which one it is until you've answered the question. The naive workaround is to put the **entire corpus** in the prompt every single time and let the model sort it out.

Below we ask about the Porter P2's battery life two ways: once with the whole corpus stuffed in, and once with only the relevant document. Watch the answer — it's the same — and then watch the token count, cost, and latency.

> **Gotcha:** With only five tiny documents the model usually still gets the right answer when you stuff everything. The problem isn't (yet) correctness — it's **waste**. You're paying to send four irrelevant documents to answer from one. That waste grows linearly with every document you add, forever.

In [ ]:
# Stuff the ENTIRE corpus into one context blob.
everything = "\n\n".join(f"=== {name} ===\n{text}" for name, text in corpus.items())

q2 = "What is the battery life of the Porter P2?"

stuffed = ask(q2, context=everything)
only_relevant = ask(q2, context=corpus["product_porter_p2.md"])

print(f"Q: {q2}\n")
print("Stuff-everything approach:")
print(f"  answer        : {stuffed['answer']}")
print(f"  prompt tokens : {stuffed['prompt_tokens']}")
print(f"  cost          : ${stuffed['cost_usd']:.6f}")
print(f"  latency       : {stuffed['latency_s']:.2f}s")
print()
print("Only-the-relevant-doc approach:")
print(f"  answer        : {only_relevant['answer']}")
print(f"  prompt tokens : {only_relevant['prompt_tokens']}")
print(f"  cost          : ${only_relevant['cost_usd']:.6f}")
print(f"  latency       : {only_relevant['latency_s']:.2f}s")
print()
waste = stuffed["prompt_tokens"] / max(only_relevant["prompt_tokens"], 1)
print(f"Same answer (~12 hours). The naive approach sent {waste:.1f}x more tokens to get it.")